# Daily Challenge – Fine-tuning BERT / XLM-RoBERTa pour la Classification de Texte

**Dataset :** `Basics of BERT and XLM-RoBERTa - PyTorch` (train.csv.zip / test.csv.zip dans le zip fourni)  
**Objectif :** Fine-tuner un modèle transformer (BERT ou XLM-RoBERTa) sur une tâche de classification multi-classes avec cross-validation stratifiée.

## Setup

In [ ]:
%pip install --quiet transformers torch scikit-learn pandas

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer, XLMRobertaTokenizer,
    BertForSequenceClassification,
    XLMRobertaForSequenceClassification,
    AdamW, get_linear_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)

## Partie 1 – Comprendre BERT et XLM-RoBERTa

**BERT (Bidirectional Encoder Representations from Transformers)**
- Architecture encoder-only bidirectionnelle
- Préentraîné avec MLM (Masked Language Modeling) et NSP (Next Sentence Prediction)
- Versions : `bert-base-uncased` (12 couches, 768 dims, 110M params), `bert-large-uncased`
- Excellent pour les tâches de compréhension : classification, NER, Q&A

**XLM-RoBERTa**
- Variante multilingue de RoBERTa (sans NSP, entraînée sur 100 langues)
- Préentraîné sur CommonCrawl filtré (2.5TB de texte)
- Versions : `xlm-roberta-base` (12 couches), `xlm-roberta-large`
- Idéal pour les tâches multilingues ou en langues à faibles ressources

## Partie 2 – Tokenisation

In [ ]:
# Chargement des tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

print(f'BERT vocab size   : {bert_tokenizer.vocab_size}')
print(f'XLM-R vocab size  : {xlmr_tokenizer.vocab_size}')
print(f'BERT special tokens: {bert_tokenizer.special_tokens_map}')
print(f'XLM-R special tokens: {xlmr_tokenizer.special_tokens_map}')

In [ ]:
# Tokenisation d'une seule phrase
sentence = "The transformer architecture has revolutionized natural language processing."

bert_enc = bert_tokenizer.encode_plus(
    sentence,
    add_special_tokens=True,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
)

print('=== BERT ===' )
print('input_ids      :', bert_enc['input_ids'])
print('attention_mask :', bert_enc['attention_mask'])
print('Décodé         :', bert_tokenizer.decode(bert_enc['input_ids']))

In [ ]:
# Tokenisation d'une paire de phrases (premise + hypothesis)
premise    = "A man is playing guitar on the street."
hypothesis = "Someone is making music outside."

xlmr_enc = xlmr_tokenizer.encode_plus(
    premise, hypothesis,
    add_special_tokens=True,
    max_length=64,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
)

print('=== XLM-RoBERTa ===' )
print('input_ids      :', xlmr_enc['input_ids'][:20], '...')
print('attention_mask :', xlmr_enc['attention_mask'][:20], '...')
print('Décodé         :', xlmr_tokenizer.decode(xlmr_enc['input_ids']))

## Partie 3 – Préparation des données

In [ ]:
# Extraction du dataset depuis le zip
ZIP_PATH    = 'Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip'
EXTRACT_DIR = 'bert_data'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Le train.csv est lui-même zippé
inner_zip = os.path.join(EXTRACT_DIR, 'Basics of BERT and XLM-RoBERTa - PyTorch', 'train.csv.zip')
with zipfile.ZipFile(inner_zip, 'r') as z:
    z.extractall(os.path.join(EXTRACT_DIR, 'Basics of BERT and XLM-RoBERTa - PyTorch'))

print('Extraction terminée.')
print(os.listdir(os.path.join(EXTRACT_DIR, 'Basics of BERT and XLM-RoBERTa - PyTorch')))

## Partie 4 – Chargement et exploration du dataset

In [ ]:
data_dir = os.path.join(EXTRACT_DIR, 'Basics of BERT and XLM-RoBERTa - PyTorch')
df = pd.read_csv(os.path.join(data_dir, 'train.csv'))

print('Shape :', df.shape)
display(df.head())
print('\nColonnes :', df.columns.tolist())
print('\nTypes :')
print(df.dtypes)

In [ ]:
# Identification de la colonne texte et de la colonne label
text_col  = df.columns[1]   # adapter selon le dataset
label_col = df.columns[-1]  # adapter selon le dataset

print(f'Colonne texte : {text_col!r}')
print(f'Colonne label : {label_col!r}')
print(f'\nDistribution des labels :')
print(df[label_col].value_counts())

num_classes = df[label_col].nunique()
print(f'\nNombre de classes : {num_classes}')

## Partie 5 – Cross-Validation Stratifiée (5 folds)

In [ ]:
# Encodage des labels en entiers
label2id = {label: idx for idx, label in enumerate(df[label_col].unique())}
id2label = {v: k for k, v in label2id.items()}

df['label_id'] = df[label_col].map(label2id)

X = df[text_col].values
y = df['label_id'].values

# StratifiedKFold : maintient la distribution des classes dans chaque fold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_folds = []
val_folds   = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    train_folds.append(train_idx)
    val_folds.append(val_idx)
    print(f'Fold {fold+1} | Train: {len(train_idx)} | Val: {len(val_idx)}')

print(f'\nDistribution vérifiée sur fold 1 :')
for label, idx in label2id.items():
    count = (y[train_folds[0]] == idx).sum()
    print(f'  {label}: {count} ({count/len(train_folds[0])*100:.1f}%)')

## Partie 6 – Dataset PyTorch + Fine-tuning BERT (Fold 1)

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer.encode_plus(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# Fine-tuning sur le Fold 1 uniquement (démo)
fold = 0
train_idx = train_folds[fold]
val_idx   = val_folds[fold]

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]

MAX_LEN    = 128
BATCH_SIZE = 16
NUM_EPOCHS = 2

tokenizer  = bert_tokenizer

train_dataset = TextClassificationDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = TextClassificationDataset(X_val,   y_val,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

print(f'Train : {len(train_dataset)} | Val : {len(val_dataset)}')

In [ ]:
# Chargement de BERT pour classification
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
)
bert_model.to(DEVICE)

optimizer = AdamW(bert_model.parameters(), lr=2e-5, eps=1e-8)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

print('Modèle prêt.')

In [ ]:
# Boucle d'entraînement
for epoch in range(NUM_EPOCHS):
    bert_model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        mask      = batch['attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Validation
    bert_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask      = batch['attention_mask'].to(DEVICE)
            labels    = batch['label'].to(DEVICE)
            logits    = bert_model(input_ids=input_ids, attention_mask=mask).logits
            preds     = logits.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())

    val_acc = accuracy_score(all_labels, all_preds)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.4f}')

print('\nRapport de classification (Fold 1) :')
print(classification_report(all_labels, all_preds,
      target_names=[id2label[i] for i in sorted(id2label)]))